# The interactive viewer

`PeakNavViewer` is an ipywidgets face on the same REST API as notebook 02 — pan and tilt
buttons, a height control, coordinates to type, display toggles, and the rendered view.

Needs `ipywidgets`, which the base package does not install:

```bash
pip install 'peaknav[jupyter]'
```

Plus what notebook 02 needs: Java, a display, and the renderer jar.

In [ ]:
from peaknav.headless import PeakNavHeadless
from peaknav.jupyter import PeakNavViewer

nav = PeakNavHeadless(46.0207, 7.7491, width=1200, height=700)

The widget is the value of the cell, so a bare expression displays it. Every button below
is one or two HTTP calls on the renderer — the widget never touches the jar or a
subprocess, which is why it works identically against a renderer someone else started.

In [ ]:
viewer = PeakNavViewer(nav, lat=46.0207, lon=7.7491,
                       bearing_deg=230, pitch_deg=-4, altitude_m=3200)
viewer

## If the controls do not appear

A cell showing `HBox(children=(Image(value=b'\xff\xd8...` instead of buttons means the
notebook was handed a widget it could not draw. Two causes:

* **the frontend has no widget support** — `pip install jupyterlab_widgets` (JupyterLab or
  Notebook 7) or `widgetsnbextension` (classic notebook), then reload the page;
* **the kernel is a different environment** from the one with ipywidgets in it. Check with
  `import sys, ipywidgets; print(sys.executable, ipywidgets.__version__)`.

A third cause was a bug here, now fixed and covered by a test: the viewer offered no widget
mimetype at all, so every frontend fell back to printing the text.

## The imagery underneath

The widget has an **Imagery** dropdown, filled from the renderer itself - so every source
the app knows is in it, and so is anything you add. Below it is a box for the URL template
of any XYZ tile server, with **Use tiles** to apply it.

The same from code, which is all the dropdown does for you - OpenStreetMap's own
cartography, draped over the mountains:

In [ ]:
viewer.set_satellite(template="https://tile.openstreetmap.org/{z}/{x}/{y}.png",
                     name="OpenStreetMap",
                     attribution="© OpenStreetMap contributors")

Every tile is re-fetched, so the picture fills in over a few seconds - press **Render** if
it still looks half-finished. Please respect the tile server's usage policy; OpenStreetMap
expects modest use and attribution, which is why it is passed above.

What the renderer already knows, and back to one of them by id:

In [ ]:
[(p["id"], p["name"]) for p in nav.providers()]

In [ ]:
viewer.set_satellite("LANDSAT")

## Driving it from code

`viewer.camera` is a plain object, usable from any cell. Because the widget cannot know
that happened, `sync_from_camera()` pulls the sliders back into line — sliders that
disagree with the picture are a small lie that wastes an afternoon.

In [ ]:
viewer.camera.go_to(45.9763, 7.6586)     # the Matterhorn itself
viewer.camera.set_altitude(4600)
viewer.camera.aim(bearing_deg=20, pitch_deg=-8)
viewer.sync_from_camera()

A slow turn, rendered step by step. Each `turn` is one `POST /camera`, each `render` one
`GET /frame`.

In [ ]:
import time

for _ in range(8):
    viewer.camera.turn(11.25)
    viewer.render()
    time.sleep(0.2)
viewer.sync_from_camera()

## Options

* `auto_render=False` — do not draw on every press; useful over a slow link, with the
  **Render** button used deliberately.
* `image_format="png"` — sharper, several times larger per frame; the default `"jpg"` is
  chosen because a frame crosses the wire on every control press.
* `width_px` — how wide the picture is shown in the notebook.

The widget also accepts a URL instead of a client, attaching to a renderer already
running:

```python
PeakNavViewer("http://127.0.0.1:41123")
```

## Shutting down

In [ ]:
nav.close()